# Pretty Formula One

This notebook is used to fetch data from certain seasons and save them into files to be uploaded into hosted database

## Race Results

In [37]:
import fastf1
import json

YEAR = 2025
ROUND_IDS = range(1, 25)

with open(f"./data/drivers_{YEAR}.json", "r") as f:
    drivers_list = json.load(f)
    
all_rounds_data = []
for ROUND_ID in ROUND_IDS:

    race_session = fastf1.get_session(YEAR, ROUND_ID, "R")
    race_session.load(laps=False, telemetry=False, weather=False)

    race_points_map = {
        f"{row['DriverId']}_{YEAR}": int(row['Points']) 
        for _, row in race_session.results.iterrows()
    }

    id_to_number_map = {
        f"{row['DriverId']}_{YEAR}": row['DriverNumber'] 
        for _, row in race_session.results.iterrows()
    }

    sprint_points_map = {}
    has_sprint = False
    try:
        sprint_session = fastf1.get_session(YEAR, ROUND_ID, "S")
        sprint_session.load(laps=False, telemetry=False, weather=False)
        has_sprint = True
        sprint_points_map = {
            row['DriverNumber']: int(row['Points']) 
            for _, row in sprint_session.results.iterrows()
        }
    except Exception:
        pass

    race_results_list = []
    for driver in drivers_list:
        d_id = driver["id"]
        r_points = race_points_map.get(d_id, 0)
        s_points = 0
        if has_sprint:
            d_number = id_to_number_map.get(d_id)
            if d_number:
                s_points = sprint_points_map.get(d_number, 0)

        race_results_list.append({
            "driver_id": d_id,
            "racePoints": r_points,
            "sprintPoints": s_points,
        })

    event_info = race_session.event
    country = event_info["Country"]
    schedule = fastf1.get_event_schedule(YEAR)
    total_rounds = len(schedule[schedule["RoundNumber"] > 0])

    round_data = {
        "id": int(event_info["RoundNumber"]),
        "year": YEAR,
        "index": int(event_info["RoundNumber"]),
        "totalRounds": total_rounds,
        "name": event_info["EventName"],
        "nameVerbose": event_info["OfficialEventName"],
        "country": country,
        "backgroundImage": f"/assets/circuits/{country.lower().replace(' ', '_')}.png",
        "results": sorted(race_results_list, key=lambda x: x["racePoints"], reverse=True),
    }

    all_rounds_data.append(round_data)
    
filename = f"./data/rounds_{YEAR}.json"

with open(filename, "w") as f:
    json.dump(all_rounds_data, f, indent=4)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...


_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '63', '12', '23', '18', '27', '16', '81', '44', '10', '22', '31', '87', '30', '5', '14', '55', '7', '6']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetchi

## All Event Names and Weekend Dates

In [ ]:
# getting all the name of the grand prix on the season
import fastf1
import json

YEAR = 2026
schedule = fastf1.get_event_schedule(YEAR)
events_dates = []
for event_name in schedule["EventName"].to_list():
    date = schedule[schedule["EventName"] == event_name]["Session1Date"].values[0]
    date = date.strftime("%Y-%m-%dT%H:%M:%S%z")
    events_dates.append({
        "name": event_name.replace("Grand Prix", "").strip(),
        "date": date,
    })
with open(f"./data/dates_{YEAR}.json", "w") as f:
    json.dump(events_dates, f, indent=4)

## Drivers

In [36]:
import fastf1
import json

# change here the info that you want to fetch (this will result on the
# list of drivers that participated in the specified year)
YEAR = 2023

session = fastf1.get_session(YEAR, 'Monaco', 'R')
session.load(laps=False, telemetry=False, weather=False)
drivers_list = []
seen_drivers = set()
for _, row in session.results.iterrows():
    driver_slug = f"{row['DriverId']}_{YEAR}"
    if driver_slug not in seen_drivers:
        drivers_list.append({
            "id": driver_slug,
            "team": row["TeamName"],
            "abbreviation": row["Abbreviation"],
            "name": row["FullName"],
            "teamLogo": f"/assets/icons/{row['TeamName'].lower().replace(' ', '-')}.png",
        })
        seen_drivers.add(driver_slug)
with open(f"./data/drivers_{YEAR}.json", "w") as f:
    json.dump(drivers_list, f, indent=4)

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '14', '31', '44', '63', '16', '10', '55', '4', '81', '77', '21', '24', '23', '22', '11', '27', '2', '20', '18']
